In [ ]:
# ============================================================
# URBAN TRAFFIC CONGESTION 7-DAY FORECASTING
# TomTom Traffic Index | Multi-City Global Study

# REQUIREMENTS: pip install pandas numpy matplotlib seaborn lightgbm shap
#               scikit-learn scipy statsmodels
# ============================================================

In [ ]:

# ── SECTION 1: IMPORTS & CONFIGURATION ──────────────────────

import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from scipy.stats import mannwhitneyu, kruskal, shapiro
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
import shap

warnings.filterwarnings('ignore')

# ── Plot styling ──
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams.update({
    'figure.dpi':      150,
    'axes.titlesize':  12,
    'axes.labelsize':  10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8,
})

# ── City colour palette ──
CITY_COLORS = {
    'Bangkok-area-Thailand':  '#E74C3C',
    'Berlin-Germany':         '#3498DB',
    'Chicago-USA':            '#2ECC71',
    'Dublin-Ireland':         '#E67E22',
    'London-UK':              '#9B59B6',
    'Los-Angeles-County-USA': '#F39C12',
    'Mexico-City-Mexico':     '#1ABC9C',
    'New-York-City-USA':      '#34495E',
    'Paris-France':           '#E91E63',
    'Sydney-Australia':       '#00BCD4',
    'Tokyo-Japan':            '#FF5722',
}

# ── Short display names (avoids broken hyphen splits) ──
CITY_SHORT = {
    'Bangkok-area-Thailand':  'Bangkok',
    'Berlin-Germany':         'Berlin',
    'Chicago-USA':            'Chicago',
    'Dublin-Ireland':         'Dublin',
    'London-UK':              'London',
    'Los-Angeles-County-USA': 'Los Angeles',
    'Mexico-City-Mexico':     'Mexico City',
    'New-York-City-USA':      'New York',
    'Paris-France':           'Paris',
    'Sydney-Australia':       'Sydney',
    'Tokyo-Japan':            'Tokyo',
}

def short(city: str) -> str:
    return CITY_SHORT.get(city, city.split('-')[0])

# ── Constants ──
FORECAST_HORIZON = 7 * 24    # 168 hours = 7 days
OUTPUT_DIR       = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# ── SECTION 2: DATA LOADING ──────────────────────────────────

def load_all_cities(data_folder='.'):
    """
    Reads every *.csv in data_folder.
    Derives city name from filename (e.g. 'Bangkok-area-Thailand.csv').
    Returns a single concatenated DataFrame with added 'city' column.
    """
    all_files = glob.glob(os.path.join(data_folder, '*.csv'))
    if not all_files:
        raise FileNotFoundError(f"No CSV files found in '{data_folder}'")

    dfs = []
    for f in all_files:
        city_name = os.path.splitext(os.path.basename(f))[0]
        try:
            tmp = pd.read_csv(f)
            tmp['city'] = city_name
            dfs.append(tmp)
            print(f"  ✓ Loaded : {city_name:40s} | {len(tmp):>7,} rows")
        except Exception as exc:
            print(f"  ✗ ERROR  : {f} → {exc}")

    combined = pd.concat(dfs, ignore_index=True)
    print(f"\n  Total rows : {len(combined):,}")
    print(f"  Cities     : {sorted(combined['city'].unique())}")
    return combined


print("=" * 60)
print("SECTION 2 — LOADING DATA")
print("=" * 60)

df_raw = load_all_cities(data_folder='traffic-index-cities')


SECTION 2 — LOADING DATA
  ✓ Loaded : Bangkok-area-Thailand                    |   4,344 rows
  ✓ Loaded : Berlin-Germany                           |   4,343 rows
  ✓ Loaded : Chicago-USA                              |   4,343 rows
  ✓ Loaded : Dublin-Ireland                           |   4,343 rows
  ✓ Loaded : London-UK                                |  60,802 rows
  ✓ Loaded : Los-Angeles-County-USA                   |   4,343 rows
  ✓ Loaded : Mexico-City-Mexico                       |   4,344 rows
  ✓ Loaded : New-York-City-USA                        |  21,715 rows
  ✓ Loaded : Paris-France                             |   4,343 rows
  ✓ Loaded : Sydney-Australia                         |  60,816 rows
  ✓ Loaded : Tokyo-Japan                              |  69,504 rows

  Total rows : 243,240
  Cities     : ['Bangkok-area-Thailand', 'Berlin-Germany', 'Chicago-USA', 'Dublin-Ireland', 'London-UK', 'Los-Angeles-County-USA', 'Mexico-City-Mexico', 'New-York-City-USA', 'Paris-France', 'S

In [ ]:
# ── SECTION 3: DATA CLEANING & STANDARDISATION ──────────────

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Rename columns to snake_case.
    2. Parse 'time' to datetime.
    3. Sort by city → region → time.
    4. Forward-fill then back-fill nulls within each city-region group.
    5. Clip congestion to [0, 100] to remove sensor artefacts.
    """
    col_map = {
        'Region index':                'region_index',
        'Region label':                'region_label',
        'Time':                        'time',
        'Speed [kmh]':                 'speed_kmh',
        'Free flow speed [kmh]':       'free_flow_speed_kmh',
        'Congestion level [%]':        'congestion_pct',
        'Travel time per 10 km [min]': 'travel_time_per_10km',
    }
    df = df.rename(columns=col_map)
    df['time'] = pd.to_datetime(df['time'])
    df = df.sort_values(['city', 'region_index', 'time']).reset_index(drop=True)

    null_counts = df.isnull().sum()
    if null_counts.any():
        print(f"\n  Nulls found:\n{null_counts[null_counts > 0]}")
    else:
        print("  No null values found.")

    numeric_cols = ['speed_kmh', 'free_flow_speed_kmh',
                    'congestion_pct', 'travel_time_per_10km']
    df[numeric_cols] = (
        df.groupby(['city', 'region_index'])[numeric_cols]
          .transform(lambda x: x.ffill().bfill())
    )
    df['congestion_pct'] = df['congestion_pct'].clip(0, 100)

    print(f"\n  Cleaned shape : {df.shape}")
    print(f"  Date range    : {df['time'].min()}  →  {df['time'].max()}")
    return df


print("\n" + "=" * 60)
print("SECTION 3 — CLEANING & STANDARDISATION")
print("=" * 60)

df = clean_data(df_raw.copy())


SECTION 3 — CLEANING & STANDARDISATION
  No null values found.

  Cleaned shape : (243240, 8)
  Date range    : 2025-01-01 00:00:00  →  2025-06-30 23:00:00


In [ ]:
# ── SECTION 4: FEATURE ENGINEERING ──────────────────────────

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Temporal features (from 'time'):
      hour, day_of_week, day_name, month, week_of_year,
      is_weekend, is_peak_hour (07-09 and 17-19)

    Lag features (within each city-region group, shifted to
    prevent leakage into the target):
      lag_1h    — congestion 1 hour ago
      lag_24h   — same hour yesterday
      lag_168h  — same hour last week

    Rolling statistic:
      rolling_24h_mean — 24-hour trailing mean (shifted by 1)

    Interaction:
      congestion_ratio — congestion relative to free-flow capacity

    NOTE: speed_kmh and travel_time_per_10km are EXCLUDED from
    model features — they are mathematically derived from
    congestion_pct and would cause direct data leakage.
    """
    df['hour']         = df['time'].dt.hour
    df['day_of_week']  = df['time'].dt.dayofweek
    df['day_name']     = df['time'].dt.day_name()
    df['month']        = df['time'].dt.month
    df['week_of_year'] = df['time'].dt.isocalendar().week.astype(int)
    df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_peak_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)

    grp = df.groupby(['city', 'region_index'])['congestion_pct']
    df['lag_1h']           = grp.shift(1)
    df['lag_24h']          = grp.shift(24)
    df['lag_168h']         = grp.shift(168)
    df['rolling_24h_mean'] = (
        grp.transform(lambda x: x.rolling(24, min_periods=1).mean()).shift(1)
    )
    df['congestion_ratio'] = (
        df['congestion_pct'] / df['free_flow_speed_kmh'].replace(0, np.nan)
    )

    print(f"  Features added. Final shape: {df.shape}")
    return df


print("\n" + "=" * 60)
print("SECTION 4 — FEATURE ENGINEERING")
print("=" * 60)

df        = add_features(df)
df_city   = df[df['region_index'] == 0].copy()   # primary region per city
cities_list = sorted(df_city['city'].unique())
n_cities  = len(cities_list)
NCOLS     = 3
NROWS     = (n_cities + NCOLS - 1) // NCOLS


SECTION 4 — FEATURE ENGINEERING
  Features added. Final shape: (243240, 20)


In [ ]:
# ── SECTION 5: EDA VISUALISATIONS ───────────────────────────

print("\n" + "=" * 60)
print("SECTION 5 — EDA VISUALISATIONS")
print("=" * 60)


SECTION 5 — EDA VISUALISATIONS  (11 plots)


In [ ]:
# ── 5.1 — City Overview: Mean Congestion + Distribution ─────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
city_avg   = df_city.groupby('city')['congestion_pct'].mean().sort_values(ascending=False)
bar_colors = [CITY_COLORS.get(c, '#888888') for c in city_avg.index]

axes[0].bar(range(len(city_avg)), city_avg.values,
            color=bar_colors, edgecolor='white')
axes[0].set_xticks(range(len(city_avg)))
axes[0].set_xticklabels([short(c) for c in city_avg.index], rotation=30, ha='right')
axes[0].set_ylabel('Average Congestion (%)')
axes[0].set_title('Mean Congestion by City')
for i, v in enumerate(city_avg.values):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', va='bottom', fontsize=7)

city_order = city_avg.index.tolist()
box_data   = [df_city[df_city['city'] == c]['congestion_pct'].values for c in city_order]
bp = axes[1].boxplot(box_data, patch_artist=True, showfliers=False, widths=0.6)
for patch, c in zip(bp['boxes'], city_order):
    patch.set_facecolor(CITY_COLORS.get(c, '#888888'))
    patch.set_alpha(0.75)
axes[1].set_xticks(range(1, len(city_order) + 1))
axes[1].set_xticklabels([short(c) for c in city_order], rotation=30, ha='right')
axes[1].set_ylabel('Congestion Level (%)')
axes[1].set_title('Congestion Distribution (IQR) by City')

plt.suptitle('PLOT 1 — City-Level Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_city_overview.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 01 — City overview")


# ── 5.2 — Hourly Pattern by City ────────────────────────────

fig, ax = plt.subplots(figsize=(14, 6))
hourly = df_city.groupby(['city', 'hour'])['congestion_pct'].mean().reset_index()

for city in cities_list:
    sub = hourly[hourly['city'] == city]
    ax.plot(sub['hour'], sub['congestion_pct'],
            label=short(city), color=CITY_COLORS.get(city, '#888888'),
            linewidth=2, marker='o', markersize=3)

ax.axvspan(7,  9,  alpha=0.08, color='red',    label='Morning Rush')
ax.axvspan(17, 19, alpha=0.08, color='orange',  label='Evening Rush')
ax.set_xticks(range(24))
ax.set_xlabel('Hour of Day (0–23)')
ax.set_ylabel('Average Congestion (%)')
ax.set_title('PLOT 2 — Hourly Congestion Pattern by City')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_hourly_pattern.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 02 — Hourly pattern")


# ── 5.3 — Day-of-Week Pattern by City ───────────────────────

fig, ax = plt.subplots(figsize=(14, 6))
DAY_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily = df_city.groupby(['city','day_name'])['congestion_pct'].mean().reset_index()
daily['day_name'] = pd.Categorical(daily['day_name'], categories=DAY_ORDER, ordered=True)
daily = daily.sort_values('day_name')

for city in cities_list:
    sub = daily[daily['city'] == city]
    ax.plot(sub['day_name'], sub['congestion_pct'],
            label=short(city), color=CITY_COLORS.get(city, '#888888'),
            linewidth=2, marker='D', markersize=5)

ax.axvspan(4.5, 6.5, alpha=0.08, color='green', label='Weekend')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Average Congestion (%)')
ax.set_title('PLOT 3 — Day-of-Week Congestion Pattern by City')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_dayofweek_pattern.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 03 — Day-of-week pattern")


# ── 5.4 — Weekday vs Weekend ─────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

wd_stats = (df_city.groupby(['city','is_weekend'])['congestion_pct']
            .mean().unstack())
wd_stats.columns = ['Weekday','Weekend']
wd_stats.index   = [short(c) for c in wd_stats.index]
wd_stats = wd_stats.sort_values('Weekday', ascending=False)
wd_stats.plot(kind='bar', ax=axes[0],
              color=['#3498DB','#E74C3C'], edgecolor='white')
axes[0].set_title('Mean Congestion: Weekday vs Weekend')
axes[0].set_ylabel('Average Congestion (%)')
axes[0].tick_params(axis='x', rotation=30)

hourly_wdwe = df_city.groupby(['is_weekend','hour'])['congestion_pct'].mean().reset_index()
for flag, label, color in [(0,'Weekday','#3498DB'),(1,'Weekend','#E74C3C')]:
    sub = hourly_wdwe[hourly_wdwe['is_weekend'] == flag]
    axes[1].plot(sub['hour'], sub['congestion_pct'],
                 label=label, color=color, linewidth=2.5)
axes[1].set_xticks(range(24))
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Average Congestion (%)')
axes[1].set_title('Hourly Profile: Weekday vs Weekend (All Cities)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('PLOT 4 — Weekday vs Weekend Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_weekday_weekend.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 04 — Weekday vs Weekend")


# ── 5.5 — Heatmap: Hour × Day-of-Week per City ──────────────

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(18, NROWS * 4))
axes = axes.flatten()

for idx, city in enumerate(cities_list):
    sub   = df_city[df_city['city'] == city]
    pivot = sub.groupby(['day_of_week','hour'])['congestion_pct'].mean().unstack()
    pivot.index = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    sns.heatmap(pivot, ax=axes[idx], cmap='YlOrRd', vmin=0, vmax=60,
                cbar_kws={'label':'Congestion %'}, xticklabels=2, linewidths=0.1)
    axes[idx].set_title(short(city), fontweight='bold')
    axes[idx].set_xlabel('Hour of Day')
    axes[idx].set_ylabel('')

for j in range(n_cities, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('PLOT 5 — Congestion Heatmap: Hour × Day of Week by City',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_heatmap_hour_dow.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 05 — Heatmap: Hour × Day-of-Week")


# ── 5.6 — Daily Average Congestion Time Series ──────────────

fig, ax = plt.subplots(figsize=(16, 6))
df_city['date'] = df_city['time'].dt.date
daily_ts = df_city.groupby(['city','date'])['congestion_pct'].mean().reset_index()
daily_ts['date'] = pd.to_datetime(daily_ts['date'])

for city in cities_list:
    sub = daily_ts[daily_ts['city'] == city]
    ax.plot(sub['date'], sub['congestion_pct'],
            label=short(city), color=CITY_COLORS.get(city, '#888888'),
            linewidth=1.2, alpha=0.85)

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.set_xlabel('Date')
ax.set_ylabel('Daily Average Congestion (%)')
ax.set_title('PLOT 6 — Daily Congestion Trend Over Time')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_daily_trend.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 06 — Daily time-series trend")


# ── 5.7 — Monthly Seasonality by City ───────────────────────

fig, ax = plt.subplots(figsize=(14, 6))
MONTH_NAMES = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly = df_city.groupby(['city','month'])['congestion_pct'].mean().reset_index()

for city in cities_list:
    sub = monthly[monthly['city'] == city]
    ax.plot([MONTH_NAMES[m] for m in sub['month']], sub['congestion_pct'],
            label=short(city), color=CITY_COLORS.get(city, '#888888'),
            linewidth=2, marker='s', markersize=4)

ax.set_xlabel('Month')
ax.set_ylabel('Average Congestion (%)')
ax.set_title('PLOT 7 — Monthly Seasonality by City')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07_monthly_pattern.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 07 — Monthly seasonality")


# ── 5.8 — Peak vs Off-Peak Horizontal Bar ───────────────────

fig, ax = plt.subplots(figsize=(12, 6))
peak_stats = (df_city.groupby(['city','is_peak_hour'])['congestion_pct']
              .mean().unstack())
peak_stats.columns = ['Off-Peak','Peak Hour']
peak_stats.index   = [short(c) for c in peak_stats.index]
peak_stats = peak_stats.sort_values('Peak Hour', ascending=True)
peak_stats.plot(kind='barh', ax=ax,
                color=['#BDC3C7','#E74C3C'], edgecolor='white')
ax.set_title('PLOT 8 — Peak vs Off-Peak Congestion by City')
ax.set_xlabel('Average Congestion (%)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08_peak_offpeak.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 08 — Peak vs Off-Peak")


# ── 5.9 — Feature Correlation Heatmap ───────────────────────

fig, ax = plt.subplots(figsize=(10, 8))
CORR_COLS = ['congestion_pct','hour','day_of_week','is_weekend',
             'is_peak_hour','free_flow_speed_kmh',
             'lag_1h','lag_24h','lag_168h','rolling_24h_mean']
corr_matrix = df_city[CORR_COLS].dropna().corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, annot_kws={'size': 8})
ax.set_title('PLOT 9 — Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/09_correlation_heatmap.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 09 — Correlation heatmap")


# ── 5.10 — Regional Intra-City Hourly Comparison ────────────

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(18, NROWS * 4))
axes = axes.flatten()

for idx, city in enumerate(cities_list):
    sub        = df[df['city'] == city]
    region_avg = sub.groupby(['region_label','hour'])['congestion_pct'].mean().reset_index()
    for region in sub['region_label'].unique():
        r_data = region_avg[region_avg['region_label'] == region]
        axes[idx].plot(r_data['hour'], r_data['congestion_pct'],
                       linewidth=1.5, label=region, alpha=0.85)
    axes[idx].set_title(short(city), fontweight='bold')
    axes[idx].set_xlabel('Hour of Day')
    axes[idx].set_ylabel('Congestion (%)')
    axes[idx].legend(fontsize=6, loc='upper left')
    axes[idx].grid(True, alpha=0.3)

for j in range(n_cities, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('PLOT 10 — Hourly Congestion by Sub-Region (Intra-City)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/10_regional_analysis.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 10 — Regional intra-city comparison")


# ── 5.11 — Violin: Congestion by Time-of-Day Bucket ─────────

def hour_bucket(h):
    if   h < 6:  return '00-06 Night'
    elif h < 10: return '06-10 Morning Rush'
    elif h < 16: return '10-16 Midday'
    elif h < 20: return '16-20 Evening Rush'
    else:        return '20-24 Night'

df_city['hour_bucket'] = df_city['hour'].apply(hour_bucket)
BUCKET_ORDER = ['00-06 Night','06-10 Morning Rush','10-16 Midday',
                '16-20 Evening Rush','20-24 Night']

fig, ax = plt.subplots(figsize=(14, 6))
sns.violinplot(data=df_city, x='hour_bucket', y='congestion_pct',
               order=BUCKET_ORDER, palette='Set2', inner='quartile', ax=ax)
ax.set_title('PLOT 11 — Congestion Distribution by Time-of-Day Bucket (All Cities)')
ax.set_xlabel('Time-of-Day Bucket')
ax.set_ylabel('Congestion Level (%)')
ax.tick_params(axis='x', rotation=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/11_violin_hour_bucket.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 11 — Violin: time-of-day distribution")

  ✓ Plot 01 — City overview
  ✓ Plot 02 — Hourly pattern
  ✓ Plot 03 — Day-of-week pattern
  ✓ Plot 04 — Weekday vs Weekend
  ✓ Plot 05 — Heatmap: Hour × Day-of-Week
  ✓ Plot 06 — Daily time-series trend
  ✓ Plot 07 — Monthly seasonality
  ✓ Plot 08 — Peak vs Off-Peak
  ✓ Plot 09 — Correlation heatmap
  ✓ Plot 10 — Regional intra-city comparison
  ✓ Plot 11 — Violin: time-of-day distribution


In [ ]:
# ── SECTION 5B: STATISTICAL ANALYSIS ────────────────────────
#
# 4 methods chosen for this data:
#
#  1. Descriptive Statistics — foundational summary per city
#     (mean, median, std, percentiles, skewness, kurtosis)
#
#  2. Mann-Whitney U Test — Weekday vs Weekend per city
#     Non-parametric (traffic data is non-normal / bimodal).
#     Tests whether weekday and weekend congestion distributions
#     differ significantly. Reports effect size (rank-biserial r)
#     to quantify HOW LARGE the difference is, not just if it exists.
#     Directly answers RQ2 (temporal drivers).
#
#  3. Kruskal-Wallis Test — Inter-city comparison
#     Non-parametric equivalent of one-way ANOVA.
#     Tests whether at least one city has a significantly different
#     congestion distribution. Directly answers RQ3 (regional differences).
#
#  4. ADF Stationarity Test — Time-series validation
#     Augmented Dickey-Fuller test per city.
#     H0: Series has a unit root (non-stationary).
#     Reject H0 (p < 0.05) → stationary → safe to use lag features
#     and LightGBM forecasting without differencing.
#     Validates the modelling approach chosen in Section 6.
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("SECTION 5B — STATISTICAL ANALYSIS")
print("=" * 60)


# ── 5B.1 — Descriptive Statistics per City ──────────────────

print("\n  [5B.1] Descriptive Statistics per City")
print("  " + "-" * 90)

desc_rows = []
for city in cities_list:
    s = df_city[df_city['city'] == city]['congestion_pct']
    desc_rows.append({
        'City':     short(city),
        'N':        int(s.count()),
        'Mean':     round(s.mean(), 2),
        'Median':   round(s.median(), 2),
        'Std':      round(s.std(), 2),
        'Min':      round(s.min(), 2),
        'P25':      round(s.quantile(0.25), 2),
        'P75':      round(s.quantile(0.75), 2),     # ← severe congestion threshold
        'P90':      round(s.quantile(0.90), 2),
        'Max':      round(s.max(), 2),
        'Skewness': round(s.skew(), 3),
        'Kurtosis': round(s.kurtosis(), 3),
    })

desc_df = pd.DataFrame(desc_rows)
print(desc_df.to_string(index=False))
desc_df.to_csv(f'{OUTPUT_DIR}/stats_01_descriptive.csv', index=False)
print("\n  ✓ Saved → stats_01_descriptive.csv")
print("  Note: P75 column = recommended binary threshold for 'severe congestion'")

# Visualise descriptive stats: Mean ± Std + P75 threshold
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors_desc = [CITY_COLORS.get(c, '#888888') for c in cities_list]
axes[0].bar(desc_df['City'], desc_df['Mean'],
            yerr=desc_df['Std'], capsize=5, color=colors_desc,
            edgecolor='white', error_kw={'ecolor': '#444', 'linewidth': 1.2})
axes[0].set_title('Mean Congestion ± Std Dev')
axes[0].set_ylabel('Congestion (%)')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(desc_df['City'], desc_df['P75'], color=colors_desc,
            edgecolor='white', alpha=0.85, label='P75 (Severe Threshold)')
axes[1].plot(range(len(desc_df)), desc_df['P90'], 'r--o',
             linewidth=1.5, markersize=5, label='P90')
axes[1].plot(range(len(desc_df)), desc_df['Median'], 'b:s',
             linewidth=1.5, markersize=5, label='Median (P50)')
axes[1].set_xticks(range(len(desc_df)))
axes[1].set_xticklabels(desc_df['City'], rotation=30, ha='right')
axes[1].set_title('Congestion Percentiles: P50 / P75 / P90')
axes[1].set_ylabel('Congestion (%)')
axes[1].legend()

plt.suptitle('STATS 1 — Descriptive Statistics by City', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/stats_01_descriptive.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot saved → stats_01_descriptive.png")


# ── 5B.2 — Mann-Whitney U: Weekday vs Weekend ───────────────
#
# Effect size — rank-biserial r:
#   r = (2U) / (n1 × n2) − 1
#   |r| < 0.1  → Negligible
#   0.1–0.3    → Small
#   0.3–0.5    → Medium
#   > 0.5      → Large

print("\n  [5B.2] Mann-Whitney U Test — Weekday vs Weekend")
print(f"  {'City':<14} {'WD Mean':>9} {'WE Mean':>9} {'Diff':>7} "
      f"{'p-value':>12} {'Sig':>5} {'Effect r':>9} {'Magnitude':>11}")
print("  " + "-" * 82)

mw_rows = []
for city in cities_list:
    sub     = df_city[df_city['city'] == city]
    wd_vals = sub[sub['is_weekend'] == 0]['congestion_pct'].dropna()
    we_vals = sub[sub['is_weekend'] == 1]['congestion_pct'].dropna()
    u_stat, p_val = mannwhitneyu(wd_vals, we_vals, alternative='two-sided')

    # Significance stars
    sig = ('***' if p_val < 0.001 else '**' if p_val < 0.01
           else '*' if p_val < 0.05 else 'ns')

    # Rank-biserial effect size
    r   = (2 * u_stat) / (len(wd_vals) * len(we_vals)) - 1
    mag = ('Large' if abs(r) > 0.5 else 'Medium' if abs(r) > 0.3
           else 'Small' if abs(r) > 0.1 else 'Negligible')

    diff = round(wd_vals.mean() - we_vals.mean(), 2)
    print(f"  {short(city):<14} {wd_vals.mean():>9.2f} {we_vals.mean():>9.2f} "
          f"{diff:>7.2f} {p_val:>12.6f} {sig:>5} {r:>9.4f} {mag:>11}")
    mw_rows.append({
        'City': short(city), 'WD_Mean': round(wd_vals.mean(), 2),
        'WE_Mean': round(we_vals.mean(), 2), 'Diff': diff,
        'U_stat': round(u_stat, 0), 'p_value': round(p_val, 6),
        'Significant': sig, 'Effect_r': round(r, 4), 'Magnitude': mag
    })

mw_df = pd.DataFrame(mw_rows)
mw_df.to_csv(f'{OUTPUT_DIR}/stats_02_mannwhitney_wdwe.csv', index=False)
print("\n  Significance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant")
print("  Effect r    : |r| > 0.5 = Large, 0.3–0.5 = Medium, 0.1–0.3 = Small")
print("  ✓ Saved → stats_02_mannwhitney_wdwe.csv")

# Visualise: effect size by city
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

mw_sorted = mw_df.sort_values('Effect_r', ascending=True)
bar_col = ['#E74C3C' if abs(r) > 0.5 else '#F39C12' if abs(r) > 0.3
           else '#3498DB' if abs(r) > 0.1 else '#BDC3C7'
           for r in mw_sorted['Effect_r']]
axes[0].barh(mw_sorted['City'], mw_sorted['Effect_r'].abs(),
             color=bar_col, edgecolor='white')
axes[0].axvline(0.1, color='#3498DB', linestyle='--', linewidth=1, label='Small (0.1)')
axes[0].axvline(0.3, color='#F39C12', linestyle='--', linewidth=1, label='Medium (0.3)')
axes[0].axvline(0.5, color='#E74C3C', linestyle='--', linewidth=1, label='Large (0.5)')
axes[0].set_title('Effect Size |r| — Weekday vs Weekend')
axes[0].set_xlabel('Rank-biserial |r|')
axes[0].legend(fontsize=8)

mw_plot = mw_df.set_index('City')[['WD_Mean','WE_Mean']].sort_values('WD_Mean', ascending=False)
mw_plot.plot(kind='bar', ax=axes[1], color=['#3498DB','#E74C3C'], edgecolor='white')
axes[1].set_title('Mean Congestion: Weekday vs Weekend')
axes[1].set_ylabel('Mean Congestion (%)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(['Weekday','Weekend'])

plt.suptitle('STATS 2 — Mann-Whitney U: Weekday vs Weekend', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/stats_02_mannwhitney.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot saved → stats_02_mannwhitney.png")


# ── 5B.3 — Kruskal-Wallis: Inter-City Comparison ────────────
#
# Tests H0: All cities have the same congestion distribution.
# Reject H0 (p < 0.05) → at least one city differs significantly.

print("\n  [5B.3] Kruskal-Wallis Test — Inter-City Congestion Comparison")
city_groups = [df_city[df_city['city'] == c]['congestion_pct'].dropna().values
               for c in cities_list]
h_stat, p_val = kruskal(*city_groups)
sig_kw = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

print(f"  H-statistic : {h_stat:.4f}")
print(f"  p-value     : {p_val:.8f}  [{sig_kw}]")
print(f"  Conclusion  : {'Significant — cities have different congestion distributions' if p_val < 0.05 else 'No significant difference across cities'}")

# Simple pairwise Mann-Whitney to identify which city pairs differ
# Using Bonferroni correction for multiple comparisons
import itertools
pairs     = list(itertools.combinations(cities_list, 2))
n_tests   = len(pairs)
alpha_adj = 0.05 / n_tests    # Bonferroni-corrected alpha

pw_rows, sig_count = [], 0
for c1, c2 in pairs:
    v1 = df_city[df_city['city'] == c1]['congestion_pct'].dropna().values
    v2 = df_city[df_city['city'] == c2]['congestion_pct'].dropna().values
    _, p = mannwhitneyu(v1, v2, alternative='two-sided')
    is_sig = p < alpha_adj
    if is_sig:
        sig_count += 1
    pw_rows.append({'City1': short(c1), 'City2': short(c2),
                    'p_value': round(p, 6),
                    'Sig_Bonferroni': 'Yes' if is_sig else 'No'})

pw_df = pd.DataFrame(pw_rows)
pw_df.to_csv(f'{OUTPUT_DIR}/stats_03_kruskal_pairwise.csv', index=False)
print(f"\n  Bonferroni α = {alpha_adj:.5f}")
print(f"  Significant city pairs : {sig_count} / {n_tests}")
print("  ✓ Saved → stats_03_kruskal_pairwise.csv")

# Visualise: pairwise significance matrix (heatmap)
pw_matrix = pd.DataFrame(index=[short(c) for c in cities_list],
                          columns=[short(c) for c in cities_list],
                          dtype=float)
for _, row in pw_df.iterrows():
    val = 1.0 if row['Sig_Bonferroni'] == 'Yes' else 0.0
    pw_matrix.loc[row['City1'], row['City2']] = val
    pw_matrix.loc[row['City2'], row['City1']] = val
np.fill_diagonal(pw_matrix.values.astype(float), np.nan)
pw_matrix = pw_matrix.astype(float)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pw_matrix, annot=True, fmt='.0f', cmap='RdYlGn',
            vmin=0, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'label': '1 = Significant Pair (Bonferroni)'},
            annot_kws={'size': 9})
ax.set_title('STATS 3 — Kruskal-Wallis Pairwise Significance Matrix\n'
             '(1 = significantly different, 0 = not significant)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/stats_03_kruskal_pairwise.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot saved → stats_03_kruskal_pairwise.png")


# ── 5B.4 — ADF Stationarity Test ────────────────────────────
#
# Augmented Dickey-Fuller (ADF) Test.
# H0 : The series has a unit root (non-stationary).
# Reject H0 if p < 0.05 → series is stationary →
# lag-based LightGBM forecasting is valid without differencing.

print("\n  [5B.4] ADF Stationarity Test — per City")
print(f"  {'City':<14} {'ADF Stat':>10} {'p-value':>12} {'Critical 5%':>12} "
      f"{'Lags':>6} {'Stationary?':>13}")
print("  " + "-" * 72)

adf_rows = []
for city in cities_list:
    series = (df_city[df_city['city'] == city]
              .sort_values('time')['congestion_pct']
              .dropna().values)
    result     = adfuller(series, autolag='AIC')
    adf_stat   = result[0]
    p_val      = result[1]
    n_lags     = result[2]
    crit_5pct  = result[4]['5%']
    stationary = 'Yes ***' if p_val < 0.001 else 'Yes *' if p_val < 0.05 else 'No'
    print(f"  {short(city):<14} {adf_stat:>10.4f} {p_val:>12.6f} "
          f"{crit_5pct:>12.4f} {n_lags:>6} {stationary:>13}")
    adf_rows.append({
        'City': short(city), 'ADF_stat': round(adf_stat, 4),
        'p_value': round(p_val, 6), 'Critical_5pct': round(crit_5pct, 4),
        'Lags_used': n_lags, 'Stationary': stationary
    })

adf_df = pd.DataFrame(adf_rows)
adf_df.to_csv(f'{OUTPUT_DIR}/stats_04_adf_stationarity.csv', index=False)
print("\n  *** p<0.001  → strongly stationary")
print("  Implication: lag features (lag_1h, lag_24h, lag_168h) are valid.")
print("  No differencing needed before modelling.")
print("  ✓ Saved → stats_04_adf_stationarity.csv")

# Visualise: ACF/PACF for first city (shows WHY lag_1h, lag_24h, lag_168h were chosen)
acf_city   = cities_list[0]
acf_series = (df_city[df_city['city'] == acf_city]
              .sort_values('time')['congestion_pct'].dropna().values)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
plot_acf(acf_series,  ax=axes[0], lags=48, alpha=0.05,
         title=f'ACF — {short(acf_city)} | Key lags: 1h (hourly), 24h (daily), 48h (2-day)')
plot_pacf(acf_series, ax=axes[1], lags=48, alpha=0.05, method='ywm',
          title=f'PACF — {short(acf_city)}')
for ax in axes:
    ax.axvline(x=24, color='red',    linestyle='--', linewidth=1.2,
               alpha=0.7, label='Lag 24h')
    ax.axvline(x=48, color='orange', linestyle='--', linewidth=1.2,
               alpha=0.7, label='Lag 48h')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'STATS 4 — ACF / PACF ({short(acf_city)}) — Justifies Lag Feature Choices',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/stats_04_acf_pacf.png', bbox_inches='tight')
plt.close()
print(f"  ✓ ACF/PACF plot saved → stats_04_acf_pacf.png  ({short(acf_city)})")

print("\n  Statistical analysis complete. Summary of outputs:")
print("    stats_01_descriptive.csv / .png")
print("    stats_02_mannwhitney_wdwe.csv / .png")
print("    stats_03_kruskal_pairwise.csv / .png")
print("    stats_04_adf_stationarity.csv")
print("    stats_04_acf_pacf.png")



SECTION 5B — STATISTICAL ANALYSIS

  [5B.1] Descriptive Statistics per City
  ------------------------------------------------------------------------------------------
       City    N  Mean  Median   Std  Min   P25   P75   P90   Max  Skewness  Kurtosis
    Bangkok 4344 37.81   37.50 28.04  0.0 10.30 58.40 77.17 100.0     0.373    -0.898
     Berlin 4343 29.43   26.60 19.47  0.4 11.00 44.90 56.18 100.0     0.498    -0.661
    Chicago 4343 24.88   23.10 17.44  0.7  8.60 36.90 49.20 100.0     0.596    -0.409
     Dublin 4343 34.67   30.80 25.35  0.8 12.20 49.50 70.30 100.0     0.696    -0.209
     London 4343 39.62   41.10 27.63  0.0 14.30 63.10 75.20 100.0     0.004    -1.231
Los Angeles 4343 30.75   27.50 23.47  0.5  8.45 46.85 65.30 100.0     0.570    -0.656
Mexico City 4344 42.12   41.10 32.03  0.0 11.00 68.80 87.77 100.0     0.194    -1.249
   New York 4343 31.32   27.10 22.82  0.0 10.75 47.15 63.88 100.0     0.680    -0.298
      Paris 4343 33.17   32.10 25.57  0.0  9.40 53.90 67

In [ ]:
# ── SECTION 6: FORECASTING MODEL ────────────────────────────
#
# Strategy  : One LightGBM Regressor per city
# Target    : congestion_pct  (regression — actual % value)
# Split     : time-based — last 7 days = test, rest = train
# Forecast  : recursive 7-day (168-step) ahead prediction

print("\n" + "=" * 60)
print("SECTION 6 — FORECASTING MODEL")
print("=" * 60)

FEATURE_COLS = [
    'hour', 'day_of_week', 'month',
    'is_weekend', 'is_peak_hour',
    'free_flow_speed_kmh',
    'lag_1h', 'lag_24h', 'lag_168h',
    'rolling_24h_mean',
]

LGBM_PARAMS = {
    'objective':         'regression',
    'metric':            'mae',
    'n_estimators':      600,
    'learning_rate':     0.04,
    'num_leaves':        63,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'verbose':          -1,
    'n_jobs':           -1,
}

results      = {}
models       = {}
forecast_dfs = {}

for city in cities_list:
    print(f"\n  ── {city} ──")

    city_df = (df_city[df_city['city'] == city]
               .dropna(subset=FEATURE_COLS + ['congestion_pct'])
               .sort_values('time')
               .reset_index(drop=True))

    # ── 6.1 — Time-Based Split ──────────────────────────────
    cutoff   = city_df['time'].max() - pd.Timedelta(days=7)
    train_df = city_df[city_df['time'] <= cutoff]
    test_df  = city_df[city_df['time'] >  cutoff]

    print(f"    Train : {train_df['time'].min()}  →  {train_df['time'].max()}  | {len(train_df):,}")
    print(f"    Test  : {test_df['time'].min()}   →  {test_df['time'].max()}   | {len(test_df):,}")

    X_train, y_train = train_df[FEATURE_COLS], train_df['congestion_pct']
    X_test,  y_test  = test_df[FEATURE_COLS],  test_df['congestion_pct']

    # ── 6.2 — Train ─────────────────────────────────────────
    model = lgb.LGBMRegressor(**LGBM_PARAMS)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=-1)],
    )
    models[city] = model

    # ── 6.3 — Evaluate ──────────────────────────────────────
    y_pred = model.predict(X_test).clip(0, 100)
    mae    = mean_absolute_error(y_test, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)

    # Robust MAPE: only computed on hours where true congestion > 1%
    # to avoid division-by-zero on overnight 0% readings
    mask = y_test.values > 1.0
    mape = (np.mean(np.abs((y_test.values[mask] - y_pred[mask])
                            / y_test.values[mask])) * 100
            if mask.sum() > 0 else np.nan)

    results[city] = {
        'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
        'R2':  round(r2, 3),  'MAPE(%)': round(mape, 1) if not np.isnan(mape) else 'N/A'
    }
    print(f"    MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}  MAPE={mape:.1f}%")

    # ── 6.4 — Recursive 7-Day Forecast ──────────────────────
    cong_history = list(city_df['congestion_pct'].values)
    freeflow_avg = float(city_df['free_flow_speed_kmh'].mean())
    last_time    = city_df['time'].max()

    forecast_rows = []
    for step in range(FORECAST_HORIZON):
        ft = last_time + pd.Timedelta(hours=step + 1)
        row = {
            'hour':               ft.hour,
            'day_of_week':        ft.dayofweek,
            'month':              ft.month,
            'is_weekend':         int(ft.dayofweek in [5, 6]),
            'is_peak_hour':       int(ft.hour in [7, 8, 9, 17, 18, 19]),
            'free_flow_speed_kmh': freeflow_avg,
            'lag_1h':   cong_history[-1]   if len(cong_history) >= 1   else np.nan,
            'lag_24h':  cong_history[-24]  if len(cong_history) >= 24  else np.nan,
            'lag_168h': cong_history[-168] if len(cong_history) >= 168 else np.nan,
            'rolling_24h_mean': (np.mean(cong_history[-24:])
                                 if len(cong_history) >= 24 else np.mean(cong_history)),
        }
        pred = float(model.predict(pd.DataFrame([row])[FEATURE_COLS]).clip(0, 100))
        cong_history.append(pred)
        forecast_rows.append({'time': ft, 'congestion_pct_forecast': pred})

    forecast_dfs[city] = pd.DataFrame(forecast_rows)
    print(f"    Forecast: {forecast_dfs[city]['time'].min()}  →  {forecast_dfs[city]['time'].max()}")


SECTION 6 — FORECASTING MODEL

  ── Bangkok-area-Thailand ──
    Train : 2025-01-08 00:00:00  →  2025-06-23 23:00:00  | 4,008
    Test  : 2025-06-24 00:00:00   →  2025-06-30 23:00:00   | 168
    MAE=2.19  RMSE=3.78  R²=0.985  MAPE=6.9%
    Forecast: 2025-07-01 00:00:00  →  2025-07-07 23:00:00

  ── Berlin-Germany ──
    Train : 2025-01-08 00:00:00  →  2025-06-23 23:00:00  | 4,007
    Test  : 2025-06-24 00:00:00   →  2025-06-30 23:00:00   | 168
    MAE=1.46  RMSE=2.11  R²=0.990  MAPE=6.3%
    Forecast: 2025-07-01 00:00:00  →  2025-07-07 23:00:00

  ── Chicago-USA ──
    Train : 2025-01-08 00:00:00  →  2025-06-23 23:00:00  | 4,007
    Test  : 2025-06-24 00:00:00   →  2025-06-30 23:00:00   | 168
    MAE=1.61  RMSE=2.19  R²=0.985  MAPE=9.0%
    Forecast: 2025-07-01 00:00:00  →  2025-07-07 23:00:00

  ── Dublin-Ireland ──
    Train : 2025-01-08 00:00:00  →  2025-06-23 23:00:00  | 4,007
    Test  : 2025-06-24 00:00:00   →  2025-06-30 23:00:00   | 168
    MAE=2.60  RMSE=4.03  R²=0.974  MAPE=

In [ ]:
# ── SECTION 7: PAST vs FUTURE FORECAST PLOTS ────────────────

print("\n" + "=" * 60)
print("SECTION 7 — PAST vs FUTURE FORECAST PLOTS")
print("=" * 60)

for city in cities_list:
    city_df     = df_city[df_city['city'] == city].sort_values('time')
    forecast_df = forecast_dfs[city]
    cutoff      = city_df['time'].max() - pd.Timedelta(days=7)
    plot_from   = city_df['time'].max() - pd.Timedelta(days=30)

    hist_plot  = city_df[city_df['time'] >= plot_from]
    test_actual = city_df[city_df['time'] > cutoff]
    test_feats  = test_actual.dropna(subset=FEATURE_COLS)
    test_pred   = models[city].predict(test_feats[FEATURE_COLS]).clip(0, 100)

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.plot(hist_plot['time'], hist_plot['congestion_pct'],
            color='#95A5A6', linewidth=1.2, label='Historical Actual', alpha=0.7)
    ax.plot(test_actual['time'], test_actual['congestion_pct'],
            color='#27AE60', linewidth=1.8, label='Test Actual (Last 7 Days)')
    ax.plot(test_feats['time'], test_pred,
            color='#F39C12', linewidth=1.8, linestyle='--', label='Test Predicted')
    ax.plot(forecast_df['time'], forecast_df['congestion_pct_forecast'],
            color='#E74C3C', linewidth=2.2, linestyle='-.', label='7-Day Forecast')

    ax.axvline(cutoff,                color='#27AE60', linestyle=':', linewidth=1.5,
               label='Train/Test Boundary')
    ax.axvline(city_df['time'].max(), color='#E74C3C', linestyle=':', linewidth=1.5,
               label='Forecast Start')
    ax.axvspan(city_df['time'].max(), forecast_df['time'].max(),
               alpha=0.04, color='#E74C3C')

    ax.set_title(f'{short(city)} — Past vs Future Congestion Forecast (7 Days)',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Congestion Level (%)')
    ax.legend(fontsize=8, loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/forecast_{city}.png', bbox_inches='tight')
    plt.close()
    print(f"  ✓ Forecast plot saved — {short(city)}")


SECTION 7 — PAST vs FUTURE FORECAST PLOTS
  ✓ Forecast plot saved — Bangkok
  ✓ Forecast plot saved — Berlin
  ✓ Forecast plot saved — Chicago
  ✓ Forecast plot saved — Dublin
  ✓ Forecast plot saved — London
  ✓ Forecast plot saved — Los Angeles
  ✓ Forecast plot saved — Mexico City
  ✓ Forecast plot saved — New York
  ✓ Forecast plot saved — Paris
  ✓ Forecast plot saved — Sydney
  ✓ Forecast plot saved — Tokyo


In [ ]:
# ── SECTION 8: EVALUATION SUMMARY ───────────────────────────

print("\n" + "=" * 60)
print("SECTION 8 — MODEL EVALUATION SUMMARY")
print("=" * 60)

results_df = (pd.DataFrame(results).T
                .reset_index().rename(columns={'index': 'City'}))
results_df['City'] = results_df['City'].apply(short)
print(results_df.to_string(index=False))
results_df.to_csv(f'{OUTPUT_DIR}/model_evaluation.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (metric, color) in enumerate(
        zip(['MAE','RMSE','R2'], ['#3498DB','#E74C3C','#2ECC71'])):
    numeric_results = results_df[results_df[metric] != 'N/A'].copy()
    numeric_results[metric] = numeric_results[metric].astype(float)
    srt = numeric_results.sort_values(metric, ascending=(metric != 'R2'))
    axes[i].barh(srt['City'], srt[metric], color=color, edgecolor='white')
    axes[i].set_title(f'{metric} by City')
    axes[i].set_xlabel(metric)
    axes[i].grid(True, alpha=0.3, axis='x')

plt.suptitle('PLOT 12 — Model Evaluation Metrics by City',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/12_evaluation_metrics.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 12 — Evaluation metrics")


SECTION 8 — MODEL EVALUATION SUMMARY
       City  MAE  RMSE    R2  MAPE(%)
    Bangkok 2.19  3.78 0.985      6.9
     Berlin 1.46  2.11 0.990      6.3
    Chicago 1.61  2.19 0.985      9.0
     Dublin 2.60  4.03 0.974     11.0
     London 3.00  4.69 0.979      9.0
Los Angeles 1.51  2.12 0.990      7.7
Mexico City 2.61  3.82 0.986     10.2
   New York 2.80  3.69 0.976     11.0
      Paris 2.77  4.15 0.982     10.7
     Sydney 1.32  1.99 0.991      6.9
      Tokyo 2.90  4.13 0.983      6.7
  ✓ Plot 12 — Evaluation metrics


In [ ]:

# ── SECTION 9: FEATURE IMPORTANCE ───────────────────────────

print("\n" + "=" * 60)
print("SECTION 9 — FEATURE IMPORTANCE (LightGBM Gain)")
print("=" * 60)

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(18, NROWS * 4))
axes = axes.flatten()

for idx, city in enumerate(cities_list):
    importance = pd.Series(
        models[city].feature_importances_, index=FEATURE_COLS
    ).sort_values(ascending=True)
    axes[idx].barh(importance.index, importance.values,
                   color='#3498DB', edgecolor='white')
    axes[idx].set_title(short(city), fontweight='bold')
    axes[idx].set_xlabel('Importance (Gain)')
    axes[idx].grid(True, alpha=0.3, axis='x')

for j in range(n_cities, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('PLOT 13 — Feature Importance by City (LightGBM)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/13_feature_importance.png', bbox_inches='tight')
plt.close()
print("  ✓ Plot 13 — Feature importance")



SECTION 9 — FEATURE IMPORTANCE (LightGBM Gain)
  ✓ Plot 13 — Feature importance


In [ ]:


# ── SECTION 10: SHAP ANALYSIS ────────────────────────────────
#
# TreeExplainer + beeswarm shows direction AND magnitude of
# each feature's contribution to individual predictions.
# Change shap_city to analyse any specific city.
# To run SHAP for ALL cities: wrap this block in a for-loop.

print("\n" + "=" * 60)
print("SECTION 10 — SHAP ANALYSIS")
print("=" * 60)

shap_city = cities_list[0]   # ← change to any city key if needed
print(f"  Running SHAP for: {short(shap_city)}")

shap_df = (df_city[df_city['city'] == shap_city]
           .dropna(subset=FEATURE_COLS)
           .sample(min(2000, len(df_city[df_city['city'] == shap_city])),
                   random_state=42))
X_shap = shap_df[FEATURE_COLS]

explainer   = shap.TreeExplainer(models[shap_city])
shap_values = explainer.shap_values(X_shap)

# Beeswarm — shows direction (positive/negative) + spread
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, feature_names=FEATURE_COLS, show=False)
plt.title(f'PLOT 14 — SHAP Beeswarm: {short(shap_city)}',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/14_shap_beeswarm.png', bbox_inches='tight')
plt.close()

# Bar — mean |SHAP| global feature importance
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False)
plt.title(f'PLOT 15 — SHAP Feature Importance (Mean |SHAP|): {short(shap_city)}',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/15_shap_bar.png', bbox_inches='tight')
plt.close()
print(f"  ✓ SHAP plots saved for {short(shap_city)}")


SECTION 10 — SHAP ANALYSIS
  Running SHAP for: Bangkok
  ✓ SHAP plots saved for Bangkok
